# Argumentation strate 6 — Acceptabilité QBF : quantificateurs par énumération, crédule et sceptique

**Navigation** : [Index](README.md) | [<< TruthMaintenance](ICT-Argumentation-TruthMaintenance-Python.ipynb) | [>> Capstone](ICT-Argumentation-Capstone-Python.ipynb) | [Version Python](ICT-Argumentation-QBFAcceptance-Python.ipynb)

**Série** : ICT-Series (strate 6, argumentation).

Ce carnet est le troisième de la famille `ICT-Argumentation-*` : après les trajectoires de croyance Dung ([`ICT-Argumentation-BeliefTrajectories-Python`](ICT-Argumentation-BeliefTrajectories-Python.ipynb)) et la mémoire de justification JTMS/ATMS (`ICT-Argumentation-TruthMaintenance-Python`), il ajoute l'angle **complexité computationnelle** : lire l'*acceptabilité* d'un argument comme une **formule booléenne quantifiée** (QBF) — et la résoudre à la main, par énumération naïve, pour voir ce que « quantifier » veut dire.

**Cadre.** Une QBF est une formule propositionnelle dont les variables sont préfixées de quantificateurs ∀ (pour tout) et ∃ (il existe). Décider si une QBF est valide est le problème canonique **PSPACE-complet** (Papadimitriou 1994) : chaque bloc de quantificateurs multiplie le travail de l'évaluateur. Notre moteur n'est PAS un solveur SOTA — c'est le contraire assumé : une **énumération exhaustive** en `2^n` qui rend chaque quantificateur visible comme une boucle. C'est précisément la leçon : le quantificateur n'est pas une notation, c'est un coût.

**Statut épistémique.** Ce carnet est une *restitution pédagogique mesurée* : chaque valeur affichée est recalculée par le code de la cellule qui l'imprime, et les contrôles `CONFORME=` rejouent les invariants à chaque exécution — aucun chiffre n'est écrit à la main. La preuve différentielle « 16/16 surfaces identiques » et les 21 gates à littéraux figés de [`ict/tests/test_qbf.py`](ict/tests/test_qbf.py) (tête `a5ac1a5d`) concernaient le **port Python** (`ict/qbf.py`) face au module EPITA `qbf_native.py` : ils ne mesurent pas cette implémentation C#. Le jumeau C# rejoue les mêmes instances (mêmes cadres, mêmes préfixes de quantificateurs, mêmes matrices) et recalcule les mêmes valeurs ; l'attestation de fin de carnet énonce précisément ce qui est vérifié ici — la parité sémantique des instances et des valeurs — et ce qui ne l'est pas.

**Organ-first — copie pédagogique déclarée.** Le moteur C# `IctCSharp` est un port déclaré du module EPITA `argumentation_analysis/agents/core/logic/qbf_native.py` (né au commit `0d2ac1b0`), enseigné par le notebook EPITA `belief_revision.ipynb` §5. Aucun moteur QBF n'existe sous forme d'organe invoquable dans ce dépôt : la seule sémantique Dung réelle du jumeau vit dans `IctCSharp.DungFramework`, qui fournit notamment `PreferredExtensions` (l'organe calcule désormais la sémantique dont la question sceptique a besoin). Les détails vivent dans la section conclusion.

## 1. Setup — les organes

Deux organes BCL : `IctCSharp.DungFramework` (cadres de Dung, labeling grounded — et extensions préférées) et `IctCSharp.QbfEngine` (AST à quantificateurs, parseur, énumérateur, acceptabilité crédule/sceptique). Convention de la série : BCL-only, déterminisme complet, CPU pur. Les sorties n'impriment que des noms d'organes — aucun chemin machine.

In [1]:
#nullable enable

using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

IctCSharp.Setup();

// Organes du carnet, reecrits sur la seule BCL (le jumeau Python importe les
// memes moteurs depuis ict/) : un moteur QBF a evaluation exhaustive et un
// moteur d'argumentation de Dung (grounded / admissible / preferred).

public static class IctCSharp
{
    // Un quantificateur (forall/exists) et sa liste de variables liees.
    public sealed record Quantifier(string Type, string[] Vars);

    // Resultat d'un check QBF, imprime au format Python (True/False) pour la
    // parite litterale des sorties avec le jumeau.
    public sealed record QbfResult(bool IsValid, string Message)
    {
        public override string ToString() => $"({PyBool(IsValid)}, '{Message}')";
    }

    // Analyse complete : formule, quantificateurs, validite + metriques
    // (nombre de quantificateurs, de variables, espace de recherche 2^n).
    public sealed record QbfAnalysis(
        string Formula,
        Quantifier[] Quantifiers,
        bool Valid,
        string Message,
        int QuantifierCount,
        int VariableCount,
        BigInteger SearchSpace);

    // AST de formule propositionnelle quantifiee. Evaluate rend la semantique
    // Tarskienne : chaque connecteur evalue ses sous-formules sous l'affectation
    // courante ; les quantificateurs enumerent exhaustivement leurs variables.
    public abstract class Formula
    {
        public abstract bool Evaluate(Dictionary<string, bool> assignment);
        public abstract HashSet<string> Variables();
    }

    public sealed class Var : Formula
    {
        public string Name { get; }

        public Var(string name) => Name = name;

        // Une variable lit sa valeur dans l'affectation fournie par le
        // quantificateur qui la lie.
        public override bool Evaluate(Dictionary<string, bool> assignment) => assignment[Name];

        public override HashSet<string> Variables() => new() { Name };

        public override string ToString() => Name;
    }

    public sealed class Not : Formula
    {
        public Formula Inner { get; }

        public Not(Formula inner) => Inner = inner;

        public override bool Evaluate(Dictionary<string, bool> assignment) => !Inner.Evaluate(assignment);

        public override HashSet<string> Variables() => Inner.Variables();

        public override string ToString() => $"!({Inner})";
    }

    public sealed class And : Formula
    {
        public Formula Left { get; }
        public Formula Right { get; }

        public And(Formula left, Formula right)
        {
            Left = left;
            Right = right;
        }

        public override bool Evaluate(Dictionary<string, bool> assignment) =>
            Left.Evaluate(assignment) && Right.Evaluate(assignment);

        public override HashSet<string> Variables() =>
            Left.Variables().Concat(Right.Variables()).ToHashSet();

        public override string ToString() => $"({Left} & {Right})";
    }

    public sealed class Or : Formula
    {
        public Formula Left { get; }
        public Formula Right { get; }

        public Or(Formula left, Formula right)
        {
            Left = left;
            Right = right;
        }

        public override bool Evaluate(Dictionary<string, bool> assignment) =>
            Left.Evaluate(assignment) || Right.Evaluate(assignment);

        public override HashSet<string> Variables() =>
            Left.Variables().Concat(Right.Variables()).ToHashSet();

        public override string ToString() => $"({Left} | {Right})";
    }

    // Implication materielle : a => b se lit !a | b.
    public sealed class Implies : Formula
    {
        public Formula Left { get; }
        public Formula Right { get; }

        public Implies(Formula left, Formula right)
        {
            Left = left;
            Right = right;
        }

        public override bool Evaluate(Dictionary<string, bool> assignment) =>
            !Left.Evaluate(assignment) || Right.Evaluate(assignment);

        public override HashSet<string> Variables() =>
            Left.Variables().Concat(Right.Variables()).ToHashSet();

        public override string ToString() => $"({Left} => {Right})";
    }

    // forall : vrai si le corps est vrai pour TOUTES les valeurs des variables
    // liees — enumeration exhaustive, le moteur est volontairement naif.
    public sealed class ForAll : Formula
    {
        public string[] Vars { get; }
        public Formula Inner { get; }

        public ForAll(string[] vars, Formula inner)
        {
            Vars = vars;
            Inner = inner;
        }

        public override bool Evaluate(Dictionary<string, bool> assignment)
        {
            foreach (Dictionary<string, bool> candidate in QbfEngine.EnumerateAssignments(Vars))
            {
                // Etendre l'affectation externe avec les valeurs liees ici :
                // les variables internes masquent les externes de meme nom.
                var extended = new Dictionary<string, bool>(assignment);
                foreach (KeyValuePair<string, bool> pair in candidate)
                {
                    extended[pair.Key] = pair.Value;
                }

                if (!Inner.Evaluate(extended))
                {
                    return false;
                }
            }

            return true;
        }

        public override HashSet<string> Variables() =>
            Vars.Concat(Inner.Variables()).ToHashSet();

        public override string ToString() => $"ForAll [{string.Join(", ", Vars)}]. ({Inner})";
    }

    // exists : vrai si le corps est vrai pour AU MOINS une valeur des
    // variables liees (dual du forall).
    public sealed class Exists : Formula
    {
        public string[] Vars { get; }
        public Formula Inner { get; }

        public Exists(string[] vars, Formula inner)
        {
            Vars = vars;
            Inner = inner;
        }

        public override bool Evaluate(Dictionary<string, bool> assignment)
        {
            foreach (Dictionary<string, bool> candidate in QbfEngine.EnumerateAssignments(Vars))
            {
                var extended = new Dictionary<string, bool>(assignment);
                foreach (KeyValuePair<string, bool> pair in candidate)
                {
                    extended[pair.Key] = pair.Value;
                }

                if (Inner.Evaluate(extended))
                {
                    return true;
                }
            }

            return false;
        }

        public override HashSet<string> Variables() =>
            Vars.Concat(Inner.Variables()).ToHashSet();

        public override string ToString() => $"Exists [{string.Join(", ", Vars)}]. ({Inner})";
    }

    // Cadre de Dung : arguments numerotes + relation d'attaque (attaquant, cible).
    public sealed record DungFramework(
        int[] Arguments,
        (int Attacker, int Target)[] Attacks);

    public static class QbfEngine
    {
        // Parseur recursif par priorite croissante : => (le plus faible),
        // puis |, puis &, puis ! unaire, puis variable. Suffisant pour les
        // formules du carnet (pas de parentheses gerees : elles n'y figurent pas).
        public static Formula ParseFormula(string source)
        {
            string text = source.Trim();

            int implication = text.IndexOf("=>", StringComparison.Ordinal);
            if (implication >= 0)
            {
                return new Implies(
                    ParseFormula(text[..implication]),
                    ParseFormula(text[(implication + 2)..]));
            }

            string[] disjunctions = text.Split('|');
            if (disjunctions.Length > 1)
            {
                Formula result = ParseFormula(disjunctions[0]);
                foreach (string part in disjunctions.Skip(1))
                {
                    result = new Or(result, ParseFormula(part));
                }

                return result;
            }

            string[] conjunctions = text.Split('&');
            if (conjunctions.Length > 1)
            {
                Formula result = ParseFormula(conjunctions[0]);
                foreach (string part in conjunctions.Skip(1))
                {
                    result = new And(result, ParseFormula(part));
                }

                return result;
            }

            if (text.StartsWith("!", StringComparison.Ordinal))
            {
                return new Not(ParseFormula(text[1..]));
            }

            return new Var(text.Trim());
        }

        // Check : emballer la formule dans ses quantificateurs puis evaluer.
        // La liste etant donnee du plus externe au plus interne, on emballe en
        // partant du DERNIER element (le plus interne) pour que le premier
        // quantificateur de la liste devienne le noeud racine.
        public static QbfResult Check(Quantifier[] quantifiers, string formula)
        {
            Formula built = ParseFormula(formula);
            for (int index = quantifiers.Length - 1; index >= 0; index--)
            {
                Quantifier quantifier = quantifiers[index];
                built = quantifier.Type == "exists"
                    ? new Exists(quantifier.Vars, built)
                    : new ForAll(quantifier.Vars, built);
            }

            bool valid = built.Evaluate(new Dictionary<string, bool>());
            string message = $"QBF {(valid ? "VALID" : "INVALID")}: {formula}";
            return new QbfResult(valid, message);
        }

        // Analyze : le check plus les metriques — l'espace de recherche 2^n
        // rappelle le cout de l'evaluation exhaustive (BigInteger : n peut
        // depasser 30 dans les experiences de passage a l'echelle).
        public static QbfAnalysis Analyze(Quantifier[] quantifiers, string formula)
        {
            QbfResult result = Check(quantifiers, formula);
            int variableCount = quantifiers.Sum(quantifier => quantifier.Vars.Length);
            return new QbfAnalysis(
                formula,
                quantifiers,
                result.IsValid,
                result.Message,
                quantifiers.Length,
                variableCount,
                BigInteger.Pow(2, variableCount));
        }

        // Enumeration exhaustive des 2^|vars| affectations : la table de
        // verite generee recursivement, variable par variable.
        public static IEnumerable<Dictionary<string, bool>> EnumerateAssignments(string[] vars)
        {
            return Enumerate(vars, 0, new Dictionary<string, bool>());
        }

        private static IEnumerable<Dictionary<string, bool>> Enumerate(
            string[] vars,
            int index,
            Dictionary<string, bool> current)
        {
            if (index == vars.Length)
            {
                yield return new Dictionary<string, bool>(current);
                yield break;
            }

            current[vars[index]] = true;
            foreach (Dictionary<string, bool> candidate in Enumerate(vars, index + 1, current))
            {
                yield return candidate;
            }

            current[vars[index]] = false;
            foreach (Dictionary<string, bool> candidate in Enumerate(vars, index + 1, current))
            {
                yield return candidate;
            }

            current.Remove(vars[index]);
        }
    }

    public static class DungEngine
    {
        // Labelling grounded : point fixe de la fonction caracteristique.
        // Tout part de « undec » ; un argument dont un attaquant est « in »
        // passe « out » ; un argument dont TOUS les attaquants sont « out »
        // passe « in ». On itere jusqu'a stabilite — le modele minimal.
        public static Dictionary<int, string> GroundedLabeling(DungFramework framework)
        {
            var labels = framework.Arguments.ToDictionary(argument => argument, _ => "undec");
            var attackers = framework.Arguments.ToDictionary(argument => argument, _ => new List<int>());
            foreach ((int attacker, int target) in framework.Attacks)
            {
                if (attackers.ContainsKey(target))
                {
                    attackers[target].Add(attacker);
                }
            }

            bool changed;
            do
            {
                changed = false;
                foreach (int argument in framework.Arguments)
                {
                    if (labels[argument] == "out")
                    {
                        continue;
                    }

                    if (attackers[argument].Any(attacker => labels[attacker] == "in"))
                    {
                        labels[argument] = "out";
                        changed = true;
                    }
                    else if (attackers[argument].All(attacker => labels[attacker] == "out")
                             && labels[argument] != "in")
                    {
                        labels[argument] = "in";
                        changed = true;
                    }
                }
            }
            while (changed);

            return labels;
        }

        // Admissible : (1) sans conflit interne — deux membres ne s'attaquent
        // pas ; (2) chaque membre est DEFENDU — tout attaquant externe est
        // contre-attaque par un membre de l'extension.
        public static bool IsAdmissible(DungFramework framework, IReadOnlyCollection<int> extension)
        {
            foreach ((int attacker, int target) in framework.Attacks)
            {
                if (extension.Contains(attacker) && extension.Contains(target))
                {
                    return false;
                }
            }

            foreach (int member in extension)
            {
                foreach ((int attacker, int target) in framework.Attacks)
                {
                    if (target != member || extension.Contains(attacker))
                    {
                        continue;
                    }

                    bool counterAttacked = framework.Attacks.Any(counter =>
                        counter.Target == attacker && extension.Contains(counter.Attacker));
                    if (!counterAttacked)
                    {
                        return false;
                    }
                }
            }

            return true;
        }

        // Preferred : enumeration naive de TOUTES les parties (2^n), filtre
        // des admissibles, puis garde des maximales pour l'inclusion. Le garde
        // fou n<=15 borne l'explosion combinatoire — ce carnet reste sur des
        // petits cadres.
        public static List<HashSet<int>> PreferredExtensions(DungFramework framework)
        {
            int count = framework.Arguments.Length;
            if (count > 15)
            {
                throw new ArgumentException(
                    $"Framework too large ({count} args) for naive enumeration",
                    nameof(framework));
            }

            var admissible = new List<HashSet<int>>();
            for (int mask = 0; mask < 1 << count; mask++)
            {
                var extension = new HashSet<int>();
                for (int index = 0; index < count; index++)
                {
                    if ((mask & (1 << index)) != 0)
                    {
                        extension.Add(framework.Arguments[index]);
                    }
                }

                if (IsAdmissible(framework, extension))
                {
                    admissible.Add(extension);
                }
            }

            return admissible
                .Where(extension => !admissible.Any(other => IsStrictSuperset(other, extension)))
                .OrderBy(extension => string.Join(",", extension.OrderBy(argument => argument)))
                .ToList();
        }

        private static bool IsStrictSuperset(HashSet<int> candidate, HashSet<int> subset)
        {
            return candidate.Count > subset.Count && subset.IsSubsetOf(candidate);
        }
    }

    // Parite d'impression avec Python : True/False et non pas true/false.
    public static string PyBool(bool value) => value ? "True" : "False";

    public static void Setup()
    {
        Console.WriteLine("organe argumentation : DungFramework (BCL)");
        Console.WriteLine("organe qbf           : QbfEngine (BCL)");
        Console.WriteLine("semantiques exposees : grounded_labeling, est_admissible, preferred_extensions");
    }
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

organe argumentation : DungFramework (BCL)


organe qbf           : QbfEngine (BCL)


semantiques exposees : grounded_labeling, est_admissible, preferred_extensions


## 2. Quantificateurs : ce que « pour tout » et « il existe » coûtent

### 2.1 Les deux cas extrêmes

La tautologie et la contradiction sont les bornes : la première valide **quelle que soit** la valeur de `x`, la seconde invalide **quelle que soit** la recherche d'un témoin. Remarquez le vocabulaire du verdict : `VALID` / `INVALID` — le moteur parle le langage des QBF, pas celui des ensembles de modèles.

In [2]:
// Illustration 1 — tautologie et contradiction
var tautologie = IctCSharp.QbfEngine.Check(
    new[] { new IctCSharp.Quantifier("forall", new[] { "x" }) },
    "x | !x");
Console.WriteLine(tautologie);

var contradiction = IctCSharp.QbfEngine.Check(
    new[] { new IctCSharp.Quantifier("exists", new[] { "x" }) },
    "x & !x");
Console.WriteLine(contradiction);

(True, 'QBF VALID: x | !x')


(False, 'QBF INVALID: x & !x')


### 2.2 L'alternance — et pourquoi l'ordre n'est pas commutatif

Le cœur du sujet n'est pas chaque quantificateur isolé mais leur **alternance**. Prenons la matrice d'équivalence `x & y | !x & !y` (lue : `x == y`). Les deux préfixes possibles à deux blocs rendent des verdicts **opposés** sur la même matrice :

- `exists x. forall y. (x == y)` — **invalide** : on choisit `x` d'abord, puis l'adversaire (`forall`) choisit `y` pour démentir. Un seul choix ne peut pas couvrir deux valeurs.
- `forall y. exists x. (x == y)` — **valide** : l'adversaire bouge d'abord, et le répondant copie. Répondre après est plus fort que jouer avant.

Cette lecture **stratégique** (le joueur existentiel contre le joueur universel) est exactement la sémantique des jeux des QBF : chaque préfixe est un ord de jeu. L'énumération naïve la matérialise — boucle imbriquée après boucle imbriquée.

In [3]:
// Illustration 2 — l'ordre des quantificateurs décide
string matrix = "x & y | !x & !y";

var exFor = IctCSharp.QbfEngine.Check(
    new[]
    {
        new IctCSharp.Quantifier("exists", new[] { "x" }),
        new IctCSharp.Quantifier("forall", new[] { "y" })
    },
    matrix);
var forEx = IctCSharp.QbfEngine.Check(
    new[]
    {
        new IctCSharp.Quantifier("forall", new[] { "y" }),
        new IctCSharp.Quantifier("exists", new[] { "x" })
    },
    matrix);

Console.WriteLine($"exists x. forall y . (x == y) -> {IctCSharp.PyBool(exFor.IsValid)}");
Console.WriteLine($"forall y. exists x . (x == y) -> {IctCSharp.PyBool(forEx.IsValid)}");

if (exFor.IsValid || !forEx.IsValid)
{
    throw new InvalidOperationException(
        $"Verdicts incorrects : exists/forall={exFor.IsValid}, forall/exists={forEx.IsValid}");
}

exists x. forall y . (x == y) -> False


forall y. exists x . (x == y) -> True


### 2.3 Trois blocs : le répondant intérieur

À trois blocs, la même logique se poursuit un cran plus loin. Sur la matrice à trois clauses `a & b | !b & c | !a & !c`, le préfixe `exists a. forall b. exists c.` est **valide** : le premier joueur pose `a = Vrai`, l'adversaire choisit `b`, et le répondant intérieur `c` **répond à `b`** (`c = Vrai` si `b = Faux`). Le quantificateur le plus interne joue en dernier — c'est lui qui dispose de l'information de tous les coups précédents.

In [4]:
using System;
using System.Globalization;

// Illustration 3 — trois blocs et coût rendu visible
var triple = IctCSharp.QbfEngine.Check(
    new[]
    {
        new IctCSharp.Quantifier("exists", new[] { "a" }),
        new IctCSharp.Quantifier("forall", new[] { "b" }),
        new IctCSharp.Quantifier("exists", new[] { "c" })
    },
    "a & b | !b & c | !a & !c");
Console.WriteLine($"{IctCSharp.PyBool(triple.IsValid)} - {triple.Message}");

// Le cout rendu visible : l'analyse QBF rapporte l'espace de recherche.
var analyse = IctCSharp.QbfEngine.Analyze(
    new[]
    {
        new IctCSharp.Quantifier("forall", new[] { "x", "y" }),
        new IctCSharp.Quantifier("exists", new[] { "z" })
    },
    "x & y & z");
Console.WriteLine(
    $"search_space = {analyse.SearchSpace.ToString(CultureInfo.InvariantCulture)}" +
    $" | variables = {analyse.VariableCount.ToString(CultureInfo.InvariantCulture)}" +
    $" | blocs = {analyse.QuantifierCount.ToString(CultureInfo.InvariantCulture)}");

True - QBF VALID: a & b | !b & c | !a & !c


search_space = 8 | variables = 3 | blocs = 2


La statistique `search_space = 8` est le **prix affiché** : trois variables, `2^3 = 8` feuilles au pire cas. C'est là que la leçon de complexité se tient : ajouter une variable **double** l'espace ; ajouter un bloc quantifié en déplace la structure. Un solveur SOTA (résolution, expansion de préfixe) fait mieux que l'énumération — mais l'énumération est la seule méthode qui montre le problème **en entier**, feuille par feuille, raison pour laquelle ce moteur pédagogique l'assume et le déclare.

Cette grille de lecture stratégique se réutilise telle quelle en théorie des jeux : un préfixe QBF est un jeu à information parfaite où les joueurs posent leurs variables à tour de rôle, le joueur ∃ cherchant à rendre la matrice vraie, le joueur ∀ à la rendre fausse. La QBF est valide exactement quand le joueur ∃ a une **stratégie gagnante** — une fonction qui, à chaque coup adverse, associe une réponse correcte. C'est pourquoi l'ordre compte : une stratégie doit répondre au coup précédent, pas le deviner. Le répondant intérieur d'une alternance `∃∀∃` voit les coups de ses deux adversaires successifs ; le quantificateur externe, lui, joue à l'aveugle. Toute la hiérarchie polynomiale (∃, ∀, ∃∀, ∀∃...) se relit ainsi : une échelle d'information décroissante pour le joueur existentiel.

### Exercice 1 — un ordre de quantificateurs qui change le verdict

**Contexte.** La section 2.2 a montré que `forall y. exists x. (x == y)` est valide et que `exists x. forall y. (x == y)` est invalide. **Objectif.** Construire une autre matrice à deux variables, différente de l'équivalence, pour laquelle `forall y. exists x.` est **valide** et `exists x. forall y.` est **invalide**. Appelez `IctCSharp.QbfEngine.Check` avec les deux préfixes, comme dans la section 2.2.

**Attention.** L'implication `exists x. forall y. M` ⟹ `forall y. exists x. M` est un théorème : le répondant peut toujours suivre le choix de l'adversaire. Ne cherchez donc pas la direction contraire.

**Indice.** Cherchez une matrice où, pour chaque valeur de `y`, il existe un `x` qui la rend vraie, mais où aucun `x` unique ne la rend vraie quelle que soit la valeur de `y`. Les opérateurs `!`, `&` et `|` suffisent.

**Critère de vérification.** Deux verdicts opposés affichés par `QbfEngine.Check`, et une phrase qui explique pourquoi le répondant intérieur gagne dans un ordre et pas dans l'autre.

In [5]:
using System;

// Exercice 1 -- matrice où l'ordre des quantificateurs décide
// TODO etudiant : choisir une matrice à deux variables (autre que l'équivalence),
// puis appeler IctCSharp.QbfEngine.Check avec les deux préfixes
// (forall y. exists x. puis exists x. forall y.) et afficher les deux verdicts.
object Exercice1OrdreSensible()
{
    Console.WriteLine("Exercice 1 a completer");
    return null!;
}

Exercice1OrdreSensible();

Exercice 1 a completer


## 3. Le parseur : précédences, et une limite assumée

La formule arrive en **chaîne** (`"x => y"`), pas en AST. Le parseur applique les précédences `! > & > | > =>` — l'implication est la plus faible. **Limite héritée de la source et enseignée ici** : pas de parenthèses. Le `ToString()` de l'AST montre la structure retenue, et avec elle le piège : `a & b | c` se lit `(a & b) | c`, jamais `a & (b | c)`.

In [6]:
using System;

// Illustration 4 — précédences du parseur
var f = IctCSharp.QbfEngine.ParseFormula("a & b | c");
Console.WriteLine($"{f.GetType().Name} -> {f}");

var g = IctCSharp.QbfEngine.ParseFormula("a => b & c");
Console.WriteLine($"{g.GetType().Name} -> {g}");

// => est la plus faible : a => (b & c), jamais (a => b) & c.
string attendue = "(a => (b & c))";
string lue = g.ToString();
Console.WriteLine($"lecture retenue : {lue} | attendue : {attendue} | CONFORME={lue == attendue}");
if (lue != attendue)
{
    throw new InvalidOperationException($"Lecture inattendue : {lue}");
}

Or -> ((a & b) | c)


Implies -> (a => (b & c))


lecture retenue : (a => (b & c)) | attendue : (a => (b & c)) | CONFORME=True


Ce choix n'est pas un bug silencieux : il est documenté dans la documentation de l'organe et épinglé par un test (`test_parseur_precedences_sans_parentheses`). En pédagogie, une limite **montrée** vaut mieux qu'une généralité cachée : l'étudiant voit qu'un parseur sans parenthèses n'est pas neutre — il *choisit* une lecture pour vous.

## 4. Acceptabilité crédule : l'énumération EST le quantificateur

Revenons à l'argumentation. La question **crédule** est : *existe-t-il* une extension admissible qui contienne l'argument `target` ? Relisez-la : c'est un **∃** — il existe un ensemble, sans conflit, qui se défend, contenant la cible. La fonction `CredulousEngine.CredulousAcceptance` ne convertit pas cette question en QBF pour la déléguer à un solveur : elle **matérialise** le ∃ en énumérant les sous-ensembles (`2^n` masques binaires croissants) et en s'arrêtant au **premier témoin**. La recherche de témoin et le quantificateur existentiel sont le même objet vu par la théorie et par le code.

Sur le *Nixon diamond* à trois arguments (a et b s'attaquent, c est libre), chaque argument est crédulement accepté — mais les témoins racontent des choses différentes :

In [7]:
#nullable enable

using System;
using System.Collections.Generic;
using System.Linq;

// Illustration 5 — acceptabilité crédule sur le Nixon
public static class CredulousEngine
{
    public sealed record CredulousResult(
        string Target,
        bool? Accepted,
        string[]? WitnessExtension,
        string Reason,
        string Method);

    public static CredulousResult CredulousAcceptance(
        string[] arguments,
        string[][] attacks,
        string target)
    {
        if (!arguments.Contains(target))
        {
            return new CredulousResult(
                target,
                false,
                null,
                $"Argument '{target}' not in framework",
                "credulous_qbf");
        }

        var attackedBy = arguments.ToDictionary(argument => argument, _ => new HashSet<string>());
        foreach (string[] attack in attacks)
        {
            if (attack.Length >= 2 && attackedBy.ContainsKey(attack[1]))
            {
                attackedBy[attack[1]].Add(attack[0]);
            }
        }

        int count = arguments.Length;
        if (count > 15)
        {
            return new CredulousResult(
                target,
                null,
                null,
                $"Framework too large ({count} args) for naive enumeration",
                "credulous_qbf");
        }

        for (int bits = 0; bits < 1 << count; bits++)
        {
            var extension = new HashSet<string>();
            for (int index = 0; index < count; index++)
            {
                if ((bits & (1 << index)) != 0)
                {
                    extension.Add(arguments[index]);
                }
            }

            if (!extension.Contains(target))
            {
                continue;
            }

            bool conflictFree = true;
            foreach (string[] attack in attacks)
            {
                if (attack.Length >= 2 && extension.Contains(attack[0]) && extension.Contains(attack[1]))
                {
                    conflictFree = false;
                    break;
                }
            }

            if (!conflictFree)
            {
                continue;
            }

            bool admissible = true;
            foreach (string member in extension)
            {
                foreach (string attacker in attackedBy[member])
                {
                    if (extension.Contains(attacker))
                    {
                        continue;
                    }

                    bool counterAttacked = attacks.Any(attack =>
                        attack.Length >= 2 && attack[1] == attacker && extension.Contains(attack[0]));
                    if (!counterAttacked)
                    {
                        admissible = false;
                        break;
                    }
                }

                if (!admissible)
                {
                    break;
                }
            }

            if (admissible)
            {
                string[] witness = extension.OrderBy(name => name, StringComparer.Ordinal).ToArray();
                return new CredulousResult(
                    target,
                    true,
                    witness,
                    $"Found admissible extension containing {target}",
                    "credulous_qbf");
            }
        }

        return new CredulousResult(
            target,
            false,
            null,
            $"No admissible extension contains {target}",
            "credulous_qbf");
    }

    public static string FormatWitness(IEnumerable<string> witness) =>
        "[" + string.Join(", ", witness.Select(name => $"'{name}'")) + "]";
}

string[] args = { "a", "b", "c" };
string[][] attacks = { new[] { "a", "b" }, new[] { "b", "a" } };

foreach (string target in args)
{
    var result = CredulousEngine.CredulousAcceptance(args, attacks, target);
    string temoin = result.WitnessExtension is null
        ? "None"
        : CredulousEngine.FormatWitness(result.WitnessExtension);
    Console.WriteLine($"cible {target} : accepte={IctCSharp.PyBool(result.Accepted == true)} temoin={temoin}");
}

// Le temoin est DETERMINISTE : premier masque binaire admissible contenant la cible.
var deterministe = CredulousEngine.CredulousAcceptance(args, attacks, "a");
string[] attendu = { "a" };
string[] lu = deterministe.WitnessExtension ?? Array.Empty<string>();
bool conforme = lu.SequenceEqual(attendu);
Console.WriteLine($"temoin attendu=['a'] | lu={CredulousEngine.FormatWitness(lu)} | CONFORME={conforme}");
if (!conforme)
{
    throw new InvalidOperationException($"Témoin non déterministe : {CredulousEngine.FormatWitness(lu)}");
}

// Refus de mesure : au-dela de 15 arguments, l'API credule refuse de mesurer
// (accepted = null) plutot que d'enumerer 2^16 sous-ensembles.
string[] tropGrandCredi = Enumerable.Range(0, 16).Select(index => $"x{index}").ToArray();
var refusCredi = CredulousEngine.CredulousAcceptance(tropGrandCredi, Array.Empty<string[]>(), "x0");
string acceptedRefusCredi = refusCredi.Accepted.HasValue ? refusCredi.Accepted.Value.ToString() : "null";
Console.WriteLine($"refus credule : accepted={acceptedRefusCredi} | raison={refusCredi.Reason}");

cible a : accepte=True temoin=['a']


cible b : accepte=True temoin=['b']


cible c : accepte=True temoin=['c']


temoin attendu=['a'] | lu=['a'] | CONFORME=True


refus credule : accepted=null | raison=Framework too large (16 args) for naive enumeration


Deux cas limites mesurés, qui distinguent la sémantique d'une intuition vite faite :

- **La chaîne défensive** — `d` est attaqué par `b`, lui-même attaqué par `a` et `c`. Intuitivement « d est attaqué, donc refusé » ; la mesure dit le contraire : `{a, d}` est admissible (`a` **défend** `d`), témoin `["a", "d"]`. L'acceptabilité crédule est une question de **coalition défensive**, pas de statut individuel.
- **Le cycle impair** — `a -> b -> c -> a` : personne n'est crédulement accepté. Chaque attaquant d'un membre du cycle est un autre membre... qui ne peut pas coexister avec lui (conflit). Le cycle pair (Nixon) laisse deux témoins singleton ; le cycle impair n'en laisse **aucun**.

In [8]:
using System;

// Illustration 6 — chaîne défensive et cycle impair
var chaine = CredulousEngine.CredulousAcceptance(
    new[] { "a", "b", "c", "d" },
    new[] { new[] { "a", "b" }, new[] { "c", "b" }, new[] { "b", "d" } },
    "d");
Console.WriteLine(
    $"chaine, cible d : {IctCSharp.PyBool(chaine.Accepted == true)} " +
    CredulousEngine.FormatWitness(chaine.WitnessExtension ?? Array.Empty<string>()));

// Cycle impair : refuse pour TOUTE cible.
var impair = CredulousEngine.CredulousAcceptance(
    new[] { "a", "b", "c" },
    new[] { new[] { "a", "b" }, new[] { "b", "c" }, new[] { "c", "a" } },
    "a");
Console.WriteLine($"cycle impair, cible a : {IctCSharp.PyBool(impair.Accepted == true)} - {impair.Reason}");

chaine, cible d : True ['a', 'd']


cycle impair, cible a : False - No admissible extension contains a


> **Encadré — la parité mesurée est un théorème nommé.** La dichotomie mesurée ci-dessus — le cycle pair (Nixon) laisse des témoins, le cycle impair `a → b → c → a` n'en laisse **aucun** — est l'obstruction de parité que la théorie des graphes a nommée : un graphe est **biparti** (ses sommets se répartissent en deux ensembles sans arête interne) si et seulement s'il ne contient **aucun cycle impair** (König). Répartir les arguments en « deux camps sans attaque interne » est exactement une 2-coloration du graphe d'attaque : le théorème dit quand cette répartition existe, et le 3-cycle en est le contre-exemple minimal — la même ligne de parité qui sépare les deux comportements mesurés par l'organe. C'est l'exemple type du principe « généraliser allège » (S. Sheydvasser, *The Power of Abstraction*, 26/09/2026).

La borne `n <= 15` n'est pas une coquetterie : à 16 arguments, `2^16 = 65536` sous-ensembles — les deux API d'acceptabilité **refusent de mesurer** (`accepted = null`, raison explicite) plutôt que de tourner en silence ; l'organe natif `PreferredExtensions`, lui, signale la même borne par une `ArgumentException`, que l'API sceptique traduit en refus nullable. Un refus visible vaut mieux qu'un temps de calcul caché : c'est la même discipline que le `RefusDeMesure` du pipeline discursif de la série.

### Exercice 2 — prédire le témoin de l'énumération

**Contexte.** La section 4 a montré que `CredulousEngine.CredulousAcceptance` renvoie le **premier** masque binaire admissible contenant la cible, les bits étant lus dans l'ordre de la liste `arguments`. **Objectif.** Construire un cadre de Dung à 4 arguments où la cible est crédulement acceptée, mais où tous les témoins admissibles ont **deux** membres (la cible et son défenseur). Prédire le témoin sur papier, puis le vérifier avec `CredulousEngine.CredulousAcceptance`.

**Indice.** Listez les attaquants de la cible. Une extension sans conflit qui contient la cible doit aussi contenir, pour chacun de ces attaquants, un argument qui l'attaque à son tour. Écrivez l'ordre des masques croissants avant de lancer le calcul.

**Critère de vérification.** La valeur de `WitnessExtension` est égale à votre prédiction. Le témoin est une liste triée (ordre alphabétique).

In [9]:
using System;

// Exercice 2 -- prédiction du témoin
// TODO etudiant : construire un cadre de Dung à 4 arguments (arguments + attacks),
// prédire le premier masque admissible contenant la cible sur papier,
// puis vérifier avec CredulousEngine.CredulousAcceptance que le WitnessExtension
// obtenu est bien le témoin prédit.
object Exercice2Temoin()
{
    Console.WriteLine("Exercice 2 a completer");
    return null!;
}

Exercice2Temoin();

Exercice 2 a completer


## 5. Acceptabilité sceptique : le ∀ passe par l'organe natif

La question **sceptique** inverse le quantificateur : `target` est-il dans **toutes** les extensions préférées (maximales pour l'inclusion) ? C'est un **∀** sur les extensions. Ici le port **consomme l'organe** : la fonction `PreferredExtensions` que ce carnet ajoute à l'organe `DungEngine` (cadres `DungFramework`) énumère les ensembles admissiaux maximaux — le mapping noms→indices vit à la frontière, la sémantique vit dans l'organe.

Sur le Nixon à trois arguments, les extensions préférées sont `{a, c}` et `{b, c}` (les singletons `{a}` et `{b}` ne sont pas maximaux : `c` — non attaqué — les complète tous les deux) :

In [10]:
#nullable enable

using System;
using System.Collections.Generic;
using System.Linq;

// Illustration 7 — sceptique sur le Nixon
public static class SkepticalEngine
{
    public sealed record SkepticalResult(
        string Target,
        bool? Accepted,
        List<string[]> PreferredExtensions,
        string Reason,
        string Method);

    public static SkepticalResult SkepticalAcceptance(
        string[] arguments,
        string[][] attacks,
        string target)
    {
        // Refus de mesure : au-dela de 15 arguments, on refuse (accepted = null)
        // au lieu de laisser l'organe natif lever une ArgumentException.
        if (arguments.Length > 15)
        {
            return new SkepticalResult(
                target,
                null,
                new List<string[]>(),
                $"Framework too large ({arguments.Length} args) for naive enumeration",
                "skeptical_qbf");
        }

        var index = arguments
            .Select((name, position) => (name, position))
            .ToDictionary(pair => pair.name, pair => pair.position);

        var edges = new List<(int Attacker, int Target)>();
        foreach (string[] attack in attacks)
        {
            if (attack.Length >= 2 && index.ContainsKey(attack[0]) && index.ContainsKey(attack[1]))
            {
                edges.Add((index[attack[0]], index[attack[1]]));
            }
        }

        var framework = new IctCSharp.DungFramework(
            arguments.Select((_, position) => position).ToArray(),
            edges.ToArray());

        List<HashSet<int>> preferred;
        try
        {
            preferred = IctCSharp.DungEngine.PreferredExtensions(framework);
        }
        catch (ArgumentException error)
        {
            return new SkepticalResult(
                target,
                null,
                new List<string[]>(),
                error.Message,
                "skeptical_qbf");
        }

        var inverse = index.ToDictionary(pair => pair.Value, pair => pair.Key);
        var preferredNames = preferred
            .Select(extension => extension
                .Select(argument => inverse[argument])
                .OrderBy(name => name, StringComparer.Ordinal)
                .ToArray())
            .ToList();

        if (preferredNames.Count == 0)
        {
            return new SkepticalResult(
                target,
                false,
                preferredNames,
                "No preferred extensions found",
                "skeptical_qbf");
        }

        bool inAll = preferredNames.All(extension => extension.Contains(target));
        string reason = inAll
            ? $"{target} is in ALL {preferredNames.Count} preferred extensions"
            : $"{target} is NOT in all preferred extensions";

        return new SkepticalResult(target, inAll, preferredNames, reason, "skeptical_qbf");
    }

    public static string FormatPreferred(IEnumerable<string[]> preferred) =>
        "[" + string.Join(", ", preferred.Select(extension =>
            "[" + string.Join(", ", extension.Select(name => $"'{name}'")) + "]")) + "]";
}

string[] args = { "a", "b", "c" };
string[][] attacks = { new[] { "a", "b" }, new[] { "b", "a" } };

var ra = SkepticalEngine.SkepticalAcceptance(args, attacks, "a");
var rc = SkepticalEngine.SkepticalAcceptance(args, attacks, "c");
Console.WriteLine($"preferred : {SkepticalEngine.FormatPreferred(ra.PreferredExtensions)}");
Console.WriteLine($"a sceptiquement accepte : {IctCSharp.PyBool(ra.Accepted == true)} - {ra.Reason}");
Console.WriteLine($"c sceptiquement accepte : {IctCSharp.PyBool(rc.Accepted == true)}");

bool conforme = (ra.Accepted == false) && (rc.Accepted == true);
Console.WriteLine($"attendu : a=False, c=True | CONFORME={conforme}");
if (!conforme)
{
    throw new InvalidOperationException(
        $"Verdicts sceptiques inattendus : a={ra.Accepted}, c={rc.Accepted}");
}

// Refus de mesure : l'API sceptique refuse explicitement (accepted = null)
// au-dela de 15 arguments, la borne de l'organe natif PreferredExtensions.
string[] tropGrandSceptique = Enumerable.Range(0, 16).Select(index => $"x{index}").ToArray();
var refusSceptique = SkepticalEngine.SkepticalAcceptance(tropGrandSceptique, Array.Empty<string[]>(), "x0");
string acceptedRefusSceptique = refusSceptique.Accepted.HasValue
    ? refusSceptique.Accepted.Value.ToString()
    : "null";
Console.WriteLine($"refus sceptique : accepted={acceptedRefusSceptique} | raison={refusSceptique.Reason}");

preferred : [['a', 'c'], ['b', 'c']]


a sceptiquement accepte : False - a is NOT in all preferred extensions


c sceptiquement accepte : True


attendu : a=False, c=True | CONFORME=True


refus sceptique : accepted=null | raison=Framework too large (16 args) for naive enumeration


L'exemple *Tweety* (a et b se contredisent, c attaque b) resserre la lecture : la seule extension préférée est `{a, c}` — `a` est cette fois **sceptiquement** accepté, parce qu'aucne extension préférée ne peut exclure son défenseur structurel. Crédule et sceptique ne sont pas deux forces d'une même question : ce sont **deux quantificateurs** sur deux collections différentes (∃ sur les admissibles, ∀ sur les préférées).

In [11]:
using System;
using System.Linq;

// Illustration 8 — sceptique sur Tweety
var rt = SkepticalEngine.SkepticalAcceptance(
    new[] { "a", "b", "c" },
    new[] { new[] { "a", "b" }, new[] { "b", "a" }, new[] { "c", "b" } },
    "a");
Console.WriteLine($"tweety preferred : {SkepticalEngine.FormatPreferred(rt.PreferredExtensions)}");
Console.WriteLine($"a sceptiquement accepte : {IctCSharp.PyBool(rt.Accepted == true)}");

bool conforme = rt.Accepted == true;
Console.WriteLine($"attendu : a sceptiquement accepte=True | CONFORME={conforme}");
if (!conforme)
{
    throw new InvalidOperationException($"Verdict sceptique inattendu : {rt.Accepted}");
}

tweety preferred : [['a', 'c']]


a sceptiquement accepte : True


attendu : a sceptiquement accepte=True | CONFORME=True


Récapitulons la dualité telle que le code la montre, colonne par colonne : la question crédule parcourt les **admissibles** (ensembles sans conflit qui se défendent — sans exigence de maximalité) et s'arrête au premier témoin ; la question sceptique parcourt les **préférées** (les admissibles maximaux, obtenus par l'organe) et exige la présence dans toutes. Entre les deux collections, la maximalité : elle ne change aucun statut sur les cas symétriques (Nixon), mais elle change tout dès qu'un argument non attaqué vient compléter les mondes possibles — `c` rejoint chaque préférée du Nixon sans jamais être mis en cause. L'agent crédule et l'agent sceptique divergent exactement là où la structure rend un monde inévitable pour l'un et contingent pour l'autre.

## 6. Le pont : théorème de Dung relu comme inclusion, et témoin négatif

Il reste à relier les deux organes du carnet précédent et celui-ci. Le théorème structurel (Dung 1995) : **l'extension grounded est incluse dans chaque extension préférée**. Le grounded — construit par point fixe du labeling (voir [`ICT-Argumentation-BeliefTrajectories-Python`](ICT-Argumentation-BeliefTrajectories-Python.ipynb)) — est le noyau minimal ; chaque préférée est un monde maximal cohérent avec ce noyau. En langage quantifié : ce qui est accepté *sans hypothèse* (∀ implicite du point fixe) survit dans *tous* les mondes maximaux.

Le bloc suivant est le **témoin négatif** du carnet : il échoue si l'organe Dung dérive, si `PreferredExtensions` diverge de la sémantique, ou si le labeling grounded change. Il rejoue l'inclusion sur quatre instances à chaque exécution.

In [12]:
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Illustration 9 — inclusion grounded ⊆ chaque préférée (4 instances)
public static class DungFormatting
{
    public static string Indices(IEnumerable<int> indices) =>
        "[" + string.Join(", ", indices.Select(index =>
            index.ToString(CultureInfo.InvariantCulture))) + "]";

    public static string Extensions(IEnumerable<HashSet<int>> extensions) =>
        "[" + string.Join(", ", extensions.Select(extension =>
            Indices(extension.OrderBy(index => index)))) + "]";
}

var cas = new (string Nom, IctCSharp.DungFramework Framework)[]
{
    ("nixon 2", new IctCSharp.DungFramework(new[] { 0, 1 }, new[] { (0, 1), (1, 0) })),
    ("tweety", new IctCSharp.DungFramework(new[] { 0, 1, 2 }, new[] { (0, 1), (1, 0), (2, 1) })),
    ("chaine a->b", new IctCSharp.DungFramework(new[] { 0, 1 }, new[] { (0, 1) })),
    ("double defense", new IctCSharp.DungFramework(new[] { 0, 1, 2 }, new[] { (0, 1), (2, 1), (1, 2) })),
};

foreach ((string nom, IctCSharp.DungFramework framework) in cas)
{
    Dictionary<int, string> labeling = IctCSharp.DungEngine.GroundedLabeling(framework);
    var groundedIn = labeling
        .Where(pair => pair.Value == "in")
        .Select(pair => pair.Key)
        .OrderBy(argument => argument)
        .ToList();
    List<HashSet<int>> prefs = IctCSharp.DungEngine.PreferredExtensions(framework);

    foreach (HashSet<int> extension in prefs)
    {
        // Theoreme : le noyau grounded survit dans CHAQUE monde maximal.
        bool inclus = groundedIn.All(argument => extension.Contains(argument));
        Console.WriteLine(
            $"instance {nom} : grounded_in={DungFormatting.Indices(groundedIn)} " +
            $"preferred={DungFormatting.Extensions(prefs)} | inclusion CONFORME={inclus}");
        if (!inclus)
        {
            throw new InvalidOperationException(
                $"Theoreme de Dung viole sur {nom} : grounded={DungFormatting.Indices(groundedIn)} " +
                $"preferee={DungFormatting.Indices(extension.OrderBy(a => a))}");
        }
    }
}

Console.WriteLine();
Console.WriteLine("Temoin negatif OK : grounded <= chaque preferred sur les 4 instances.");

instance nixon 2 : grounded_in=[] preferred=[[0], [1]] | inclusion CONFORME=True


instance nixon 2 : grounded_in=[] preferred=[[0], [1]] | inclusion CONFORME=True


instance tweety : grounded_in=[0, 2] preferred=[[0, 2]] | inclusion CONFORME=True


instance chaine a->b : grounded_in=[0] preferred=[[0]] | inclusion CONFORME=True


instance double defense : grounded_in=[0, 2] preferred=[[0, 2]] | inclusion CONFORME=True


Temoin negatif OK : grounded <= chaque preferred sur les 4 instances.


Lecture du tableau : le noyau grandit avec la structure défensive (vide pour le Nixon pur — deux `undec` ; `{0, 2}` pour Tweety), et **chaque** préférée le contient. La réciproque est fausse en général — l'intersection des préférées peut **strictement** dépasser le grounded : c'est précisément l'objet du troisième exercice.

### Exercice 3 — quand l'intersection des préférées dépasse le grounded

**Contexte.** La section 6 a établi que le grounded est inclus dans chaque extension préférée, et a annoncé que la réciproque est fausse. **Objectif.** Construire un cadre de Dung à 4 arguments dont l'intersection des extensions préférées contient au moins un argument qui n'est pas `in` dans le labeling grounded. Mesurez-le avec `IctCSharp.DungFramework`, `IctCSharp.DungEngine.GroundedLabeling` et `IctCSharp.DungEngine.PreferredExtensions`, puis calculez l'écart avec des ensembles (`HashSet<int>`).

**Indice.** Le grounded ne tranche rien sur deux attaquants qui s'excluent mutuellement. Si un argument `d` est attaqué par un argument `c` lui-même attaqué par ces deux attaquants, chaque extension préférée peut défendre `d` en choisissant **l'un** d'eux. Vérifiez que `d` appartient alors à toutes les préférées sans appartenir au grounded.

**Critère de vérification.** L'écart `intersection des préférées - ensemble in du grounded` est non vide, et son contenu est affiché avec les indices des arguments.

In [13]:
using System;

// Exercice 3 -- écart grounded / sceptique
// TODO etudiant : construire un cadre de Dung à 4 arguments, calculer le labeling
// grounded (IctCSharp.DungEngine.GroundedLabeling), les extensions préférées
// (IctCSharp.DungEngine.PreferredExtensions), puis afficher l'écart
// (intersection des préférées - arguments 'in' du grounded) avec les indices.
object Exercice3EcartGroundedSceptique()
{
    Console.WriteLine("Exercice 3 a completer");
    return null!;
}

Exercice3EcartGroundedSceptique();

Exercice 3 a completer


## À retenir

*Ce que le carnet a établi. (1) Les quantificateurs booléens se lisent stratégiquement — ordre de jeu, répondant intérieur — et l'ordre n'est pas commutatif (§2). (2) Un parseur à précédences sans parenthèses fait des choix de lecture qu'il faut savoir voir (§3). (3) L'acceptabilité crédule est un ∃ **matérialisé** par l'énumération des coalitions défensives — la chaîne défensive et le cycle impair sont les deux cas qui cassent l'intuition individuelle (§4). (4) L'acceptabilité sceptique est un ∀ sur les extensions préférées, calculées par l'organe natif Dung que ce carnet dote de `PreferredExtensions` (§5). (5) Le théorème grounded ⊆ chaque préférée relie ce carnet au carnet Dung précédent — témoin négatif rejoué à chaque exécution (§6).*

## Vérifiez votre compréhension

Les trois exercices du carnet, chacun bornant une marche du quantificateur :

- **Exercice 1** — un ordre de quantificateurs qui change le verdict (§2).
- **Exercice 2** — prédire le témoin de l'énumération crédule (§4).
- **Exercice 3** — quand l'intersection des préférées dépasse le grounded (§6).

**Copie pédagogique déclarée** (règle `organ-first-implementation`) : l'organe C# `QbfEngine` porte la sémantique du module EPITA `qbf_native.py` (tête `a5ac1a5d`, commit fondateur `0d2ac1b0`), divergences déclarées dans la documentation de l'organe — l'acceptabilité sceptique consomme l'organe `DungEngine` au lieu de `dung_native` ; refus uniformisé `accepted = null` (les deux API d'acceptabilité renvoient un refus nullable, l'organe natif signalant la borne `n <= 15` par une `ArgumentException` que l'API sceptique traduit en refus) ; documentation en français. L'exécution différentielle « 16/16 surfaces identiques » et les 21 gates à littéraux figés de `ict/tests/test_qbf.py` mesurent le **port Python** (`ict/qbf.py`) face au module EPITA : ce carnet ne les revendique pas. Ce qu'il établit, c'est le rejeu des mêmes instances et le recalcul des mêmes valeurs à chaque exécution — les cellules 16 et 23 affichent en outre le refus de mesure des deux API d'acceptabilité. La fonction `PreferredExtensions` est une **extension de l'organe natif** (pas une copie) : la série demande, l'organe calcule.

**Attestation de parité (Python ↔ C#).** Le jumeau C# rejoue exactement les mêmes instances que le jumeau Python : mêmes arguments, mêmes attaques, mêmes préfixes de quantificateurs, mêmes matrices, mêmes verdicts et mêmes témoins. La parité est **sémantique**, non textuelle : les surfaces se lisent ligne à ligne sur les sorties committées des deux carnets, avec des écarts de forme assumés — les cellules 13, 16, 23 et 25 ajoutent à côté des valeurs partagées une ligne de contrôle `CONFORME=` ; la cellule 2 nomme les organes `DungFramework` / `QbfEngine` là où le jumeau Python nomme ses modules ; la cellule 28 imprime une ligne par extension préférée, d'où deux lignes pour le Nixon (deux préférées) et une seule pour les autres instances. Surfaces partagées : cellule 2 (organes), 4 (tautologie / contradiction), 6 (ordre des quantificateurs), 8 (trois blocs et `search_space`), 13 (précédences du parseur), 16 (crédule sur le Nixon), 18 (chaîne défensive et cycle impair), 23 (sceptique sur le Nixon), 25 (sceptique sur Tweety), 28 (inclusion grounded ⊆ chaque préférée sur les quatre instances). Aucun générateur aléatoire n'intervient : le déterminisme est total.

## Pour aller plus loin

**Références.** Dung, P. M. (1995). *On the Acceptability of Arguments and its Fundamental Role in Nonmonotonic Reasoning, Logic Programming, and n-Person Games*. Artificial Intelligence 77. · Papadimitriou, C. (1994). *Computational Complexity* — QBF et la hiérarchie polynomiale. · Pollock, J. (1987). *Defeasible Reasoning* — le Nixon diamond. · Source distillée : `2025-Epita-Intelligence-Symbolique`, `argumentation_analysis/agents/core/logic/qbf_native.py`.

- **[ICT-Argumentation-TruthMaintenance — la révision des croyances](ICT-Argumentation-TruthMaintenance-Python.ipynb)** : la couche révision du substrat argumentation — du JTMS à l'ATMS.
- **[ICT-0 — Cadrage de la série](ICT-0-Framing.md)** : le principe directeur de la série ICT.

**Suivant — strate 6** : [→ ICT-Argumentation-TruthMaintenance-Python](ICT-Argumentation-TruthMaintenance-Python.ipynb) — la couche révision des croyances (JTMS/ATMS), troisième carnet de la strate avant le capstone.

> Carnet issu de l'EPIC #4588.